In [4]:
!pip install qiskit==1.4.3
!pip install qiskit_nature==0.7.2
!pip install qiskit_aer==0.17.1
!pip install pyscf
!pip install rdkit

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 39.6 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.7/22.7 MB 39.1 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: symengine
    Found existing installation: symengine 0.14.1
    Uninstalling symengine-0.14.1:
      Successfully uninstalled symengine-0.14.1
  Attempting uninstall: qiskit
    Found existing installation: qiskit 2.2.3
    Uninstalling qiskit-2.2.3:
      Successfully uninstalled qiskit-2.2.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
qiskit-nature 0.7.2 requires qiskit-algorithms>=0.2.1, which is not installed.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 327.8/327.8 kB 9.7 MB/s eta 0:00:00
  Using cached qiskit_aer-0.17.1-cp310-cp310-macosx_11_0_arm64.whl (2.2 MB)


In [1]:
from rdkit import Chem
from rdkit.Chem import AllChem

def cap_structure(input_pdb, output_pdb):
    """
    Reads a PDB file, identifies and caps dangling bonds with hydrogen atoms,
    and saves the capped structure to a new PDB file.
    """
    # Read the molecule from the PDB file, sanitizing it to fix valency issues
    mol = Chem.MolFromPDBFile(input_pdb, sanitize=True, removeHs=False)

    if mol is None:
        print(f"Error: Could not read molecule from {input_pdb}")
        return

    # Add hydrogens to all atoms to ensure full valency
    mol = Chem.AddHs(mol, explicitOnly=True)

    # Save the new molecule with the added hydrogens
    writer = Chem.PDBWriter(output_pdb)
    writer.write(mol)
    writer.close()

    print(f"Structure from '{input_pdb}' has been capped and saved to '{output_pdb}'")

if __name__ == "__main__":
    input_filea = "res_17_21_capped.pdb"
    output_filea = "res_17_21_cappeda.pdb"
    cap_structure(input_filea, output_filea)
    input_filew = "wildtype.pdb"
    output_filew = "res_17_21_cappedw.pdb"
    cap_structure(input_filew, output_filew)

OSError: Bad input file res_17_21_capped.pdb

In [ ]:
from rdkit import Chem
def convert_pdb_to_xyz(input_pdb, output_xyz):
    """
    Converts a PDB file to an XYZ file using RDKit.
    """
    # Read the molecule from the PDB file
    mol = Chem.MolFromPDBFile(input_pdb, sanitize=False, removeHs=False)

    if mol is None:
        print(f"Error: Could not read molecule from {input_pdb}")
        return

    # Create an XYZ file writer
    writer = Chem.PDBWriter(output_xyz)
    writer.write(mol)
    writer.close()


    with open(output_xyz, 'w') as f:
        # Write the number of atoms
        f.write(f"{mol.GetNumAtoms()}\n\n")
        # Write each atom's symbol and coordinates
        for atom in mol.GetAtoms():
            pos = mol.GetConformer().GetAtomPosition(atom.GetIdx())
            symbol = atom.GetSymbol()
            f.write(f"{symbol}\t{pos.x:.4f}\t{pos.y:.4f}\t{pos.z:.4f}\n")

    print(f"Successfully converted '{input_pdb}' to '{output_xyz}'")

if __name__ == "__main__":
    input_filea = "res_17_21_cappeda.pdb"
    output_filea = "res_17_21_cappeda.xyz"
    convert_pdb_to_xyz(input_filea, output_filea)
    input_filew = "wildtype.pdb"
    output_filew = "res_17_21_cappedw.xyz"
    convert_pdb_to_xyz(input_filew, output_filew)

Successfully converted 'res_17_21_cappeda.pdb' to 'res_17_21_cappeda.xyz'
Successfully converted 'wildtype.pdb' to 'res_17_21_cappedw.xyz'


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver, VQE
from qiskit_algorithms.optimizers import COBYLA, L_BFGS_B, SPSA, SLSQP
from qiskit_nature.second_q.transformers import FreezeCoreTransformer, ActiveSpaceTransformer
from qiskit_nature.second_q.formats.molecule_info import MoleculeInfo as Molecule
from qiskit_nature.second_q.mappers import ParityMapper
from qiskit_nature.units import DistanceUnit
from qiskit.circuit.library import TwoLocal
from qiskit_nature.second_q.circuit.library import UCCSD, HartreeFock
from qiskit_nature.second_q.drivers import PySCFDriver, MethodType
from qiskit_nature.second_q.algorithms import GroundStateEigensolver
from qiskit.circuit.library import EfficientSU2
from qiskit_aer import AerSimulator, Aer
from qiskit_aer.noise import NoiseModel
from qiskit_aer.primitives import EstimatorV2, Estimator
import qiskit_nature.settings
from pyscf import solvent, gto, scf
from pyscf.solvent import ddCOSMO

qiskit_nature.settings.use_pauli_sum_op = False

In [ ]:
import qiskit.utils
import qiskit_nature.utils
import qiskit_aer.utils

print(f"qiskit version: {qiskit.__version__}")
print(f"qiskit-nature version: {qiskit_nature.__version__}")
print(f"qiskit-aer version: {qiskit_aer.__version__}")

In [ ]:
from qiskit_nature.second_q.drivers import PySCFDriver, MethodType
from qiskit_nature.second_q.transformers import ActiveSpaceTransformer
from qiskit_nature.second_q.mappers import ParityMapper, JordanWignerMapper
from qiskit_nature.second_q.problems import ElectronicStructureProblem
from pyscf import gto, scf, solvent
from pyscf.tools import cubegen

def log_orbital_energies(name, orbital_energies, orbital_occupations, log_list, orbitals=[2,3,4,5]):
    for i in orbitals:
        log_list.append({
            "fragment": name,
            "orbital_index": i,
            "energy": orbital_energies[i],
            "occupation": orbital_occupations[i]
        })
    print(log_list)
    return log_list

def create_abeta_driver(atom_string, charge, spin, name, basis='sto-3g'):
    """
    Creates and returns a Qiskit-compatible ElectronicStructureProblem.
    Solvent effects (ddCOSMO) are applied manually in PySCF but not captured in Qiskit integrals.
    """
    # Manual PySCF run with ddCOSMO (for reference or logging)
    mol = gto.Mole()
    mol.atom = atom_string
    mol.unit = 'Angstrom'
    mol.basis = basis
    mol.charge = charge
    mol.spin = spin
    mol.build()

    mf = scf.RHF(mol).ddCOSMO()
    mf.with_solvent.eps = 80.0
    mf.kernel(level_shift=0.2)
    print("Total RHF energy:", mf.e_tot, "Hartree")
    mo_coeff1 = mf.mo_coeff.copy()
    mo_occ1 = mf.mo_occ.copy()
    
    orbital1s = [150, 151, 152, 153]
    for i in orbital1s:
        cubegen.orbital(mol, f'orbital{i}_{name}.cube', mo_coeff1[:,i])


    driver = PySCFDriver(
        atom=atom_string,
        unit=DistanceUnit.ANGSTROM,
        basis=basis,
        charge=charge,
        spin=spin,
        method=MethodType.RHF
    )
    problem = driver.run()

    return problem

def get_homo_lumo_orbitals(problem, window=2):
    occ = problem.orbital_occupations
    homo_index = max(i for i, occ_val in enumerate(occ) if occ_val > 0)
    lumo_index = homo_index + 1
    return list(range(homo_index - window + 1, lumo_index + window))

def get_qubit_op(xyz_file_content, name):
    # Parse XYZ content
    lines = xyz_file_content.strip().split('\n')
    atom_data_string = '\n'.join(lines[2:]) if len(lines) > 2 else xyz_file_content.strip()

    current_charge = 0
    current_spin = 1

    # Step 1: Create initial problem
    problem = create_abeta_driver(atom_data_string, current_charge, current_spin, name)
    problem = create_abeta_driver(atom_data_string, current_charge, current_spin, name)


    for i, (occ, e) in enumerate(zip(problem.orbital_occupations, problem.orbital_energies)):
        print(f"[{name}] Orbital {i}: occ={occ:.2f}, energy={e:.6f} Hartree")

    orbital_log = []

    log_orbital_energies(name, problem.orbital_energies, problem.orbital_occupations, orbital_log)


    print(f"[{name}] Initial num_particles: {problem.num_particles}")
    print(f"[{name}] Initial num_spatial_orbitals: {problem.num_spatial_orbitals}")
    ansatz = TwoLocal(rotation_blocks='ry', entanglement_blocks='cz', reps=1)
    vqe_solver = VQE(EstimatorV2(), ansatz, SLSQP())
    solver = GroundStateEigensolver(JordanWignerMapper(), vqe_solver)
    # Step 2: Determine active orbitals
    if name == "wt":
       # Step 2: Determine active orbitals
        #temp_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=4, active_orbitals=list(range(4)))
        #temp_problem = temp_transformer.transform(problem)
        #temp_result = solver.solve(temp_problem)
        #active_orbitals = get_homo_lumo_orbitals(temp_problem, window=2)

        active_orbitals = [150, 151, 152, 153] #wt hardcoded
    else:
       # Step 2: Determine active orbitals
        #temp_transformer = ActiveSpaceTransformer(num_electrons=4, num_spatial_orbitals=4, active_orbitals=list(range(4)))
        #temp_problem = temp_transformer.transform(problem)
        #temp_result = solver.solve(temp_problem)
        #active_orbitals = get_homo_lumo_orbitals(temp_problem, window=2)
       active_orbitals = [150, 151, 152, 153]  # Arctic mutant hardcoded

    print(f"[{name}] Active orbitals selected: {active_orbitals}")

    # Step 3: Apply final transformer
    transformer = ActiveSpaceTransformer(
        num_electrons=4,
        num_spatial_orbitals=4,
        active_orbitals=active_orbitals
    )
    problem = transformer.transform(problem)

    # Step 4: Map to qubit Hamiltonian
    num_particles = problem.num_particles
    num_spatial_orbitals_transformed = problem.num_spatial_orbitals
    print(f"[{name}] Transformed num_particles: {num_particles}")
    print(f"[{name}] Transformed num_spatial_orbitals: {num_spatial_orbitals_transformed}")

    mapper = JordanWignerMapper()
    hamiltonian = mapper.map(problem.second_q_ops()[0])
    print(f"[{name}] Hamiltonian qubits: {hamiltonian.num_qubits}")

    return hamiltonian, num_particles, num_spatial_orbitals_transformed, problem, mapper


In [ ]:
def read_xyz_file(filepath, charge, mult):
    with open(filepath, 'r') as f:
        lines = f.readlines()[2:]  # Skip atom count and comment
    mol_lines = [f"{charge} {mult}"]
    for line in lines:
        parts = line.split()
        if len(parts) < 4:
            continue
        atom = parts[0]
        x, y, z = map(float, parts[1:4])
        mol_lines.append(f"{atom} {x:.6f} {y:.6f} {z:.6f}")
    return "\n".join(mol_lines)

wt_xyz_content = read_xyz_file("res_17_21_cappedw.xyz", 0, 1)
arctic_xyz_content = read_xyz_file("res_17_21_cappeda.xyz", 0, 1)
print(wt_xyz_content)
print(f"Arctic: {arctic_xyz_content}")

In [ ]:

wt_hamiltonian, wt_particles, wt_orbitals, wt_problem, wt_mapper = get_qubit_op(wt_xyz_content, name="wt")
arctic_hamiltonian, arctic_particles, arctic_orbitals, arctic_problem, arctic_mapper = get_qubit_op(arctic_xyz_content, name="mut")

def run_vqe_with_plot(hamiltonian, problem, mapper, num_particles, num_orbitals, label=""):
    # Initialize lists to store convergence data
    energy_values = []
    iteration_counts = []
    
    # Callback function to track convergence
    def callback(eval_count, parameters, mean, std):
        energy_values.append(mean)
        iteration_counts.append(eval_count)
        print(f"{label} - Iteration: {eval_count}, Energy: {mean}")
    
    # Set up ansatz 
    num_qubits = hamiltonian.num_qubits
    init_state = HartreeFock(num_orbitals, num_particles, mapper)
    ansatz_body = TwoLocal(
        num_qubits=num_qubits,
        rotation_blocks=['ry', 'rz'],
        entanglement_blocks='cx',
        entanglement='linear',
        reps=2,
        insert_barriers=True,
    )
    ansatz = QuantumCircuit(num_qubits)
    ansatz.append(init_state, list(range(num_qubits)))
    ansatz.append(ansatz_body, list(range(num_qubits)))
    ansatz.draw(output='text', fold=80, plot_barriers=False, reverse_bits=True)
    print(ansatz.draw())
    
    # Run VQE with callback
    vqe = VQE(
        estimator=Estimator(),
        ansatz=ansatz,
        optimizer=SLSQP(maxiter=100),
        callback=callback
    )
    result = vqe.compute_minimum_eigenvalue(hamiltonian)
    interpreted_result = problem.interpret(result)
    
    # Return both the result and convergence data
    return {
        'result': interpreted_result,
        'energies': energy_values,
        'iterations': iteration_counts,
        'final_energy': interpreted_result.total_energies[0]
    }

# Run VQE for both systems
wt_data = run_vqe_with_plot(wt_hamiltonian, wt_problem, wt_mapper, wt_particles, wt_orbitals, label="WT")
arctic_data = run_vqe_with_plot(arctic_hamiltonian, arctic_problem, arctic_mapper, arctic_particles, arctic_orbitals, label="Arctic")

# Create convergence plot
plt.figure(figsize=(10, 6))
plt.plot(wt_data['iterations'], wt_data['energies'], 'b-', label='WT', linewidth=2)
plt.plot(arctic_data['iterations'], arctic_data['energies'], 'r-', label='Arctic', linewidth=2)

# Add horizontal lines for final energies
#plt.axhline(y=wt_data['final_energy'], color='b', linestyle='--', alpha=0.5)
#plt.axhline(y=arctic_data['final_energy'], color='r', linestyle='--', alpha=0.5)

# Format plot
plt.xlabel('Iteration', fontsize=12)
plt.ylabel('Expectation Energy (hartree)', fontsize=12)
plt.title('VQE Convergence: WT vs Arctic', fontsize=14)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()


plt.savefig('vqe_convergence_comparison1.png', dpi=300)
plt.show()

# Compare final energies
print(f"\nFinal Energies:")
print(f"WT:     {wt_data['final_energy']} hartree")
print(f"Arctic: {arctic_data['final_energy']} hartree")
print(f"ΔE:     {arctic_data['final_energy'] - wt_data['final_energy']} hartree")

In [ ]:
import mdtraj as md

traj1 = md.load_pdb("res_17_21_cappedw.pdb")
traj2 = md.load_pdb("res_17_21_cappeda.pdb")

backbone_atoms = ['N', 'CA', 'C']
backbone_indices = [atom.index for atom in traj1.topology.atoms if atom.name in backbone_atoms]

rms = md.rmsd(traj2, traj1, atom_indices=backbone_indices)
print(rms)